## GROUP PROJECT PROPOSAL | DATA CLEANING
#### Group 6 | Phoebe Zhao (PYZ2001) | Francesco Biedermann (FB2709) | Brian Hsu (CH4004)
#### APAN5205 Applied Machine Learning 2 | Prof. Andrew Assing

### Step 1 |  Loading the Raw Data
The 2024 Loan/Application Register dataset published by the Consumer Financial Protection Bureau contains over 12.2 million mortgage application records. Due to the file's size, loading the entire dataset into memory at once was not feasible. To address this, we read the file in sequential chunks of 500,000 rows each, filtering each chunk to retain only records with an action_taken value of:

- 1 (loan originated)
- 2 (application approved but not accepted), or
- 3 (application denied)

These three values represent clear lender decisions, whereas other action codes such as 4 (application withdrawn), 5 (file closed for incompleteness) and 6 (purchased loan) do not reflect a definitive approval or denial by the lender and were therefore excluded. This filtering reduced the dataset from ~12.3 million rows to ~8.7 million rows across 99 columns.

In [1]:
# Step 1: Load the Raw Data
# The full 2024 MLAR has 12+ million rows and is too large to load at once. We read it in chunks of 500k rows, 
# filter to only the action_taken values we need (1, 2, 3), and concatenate the filtered pieces.

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Read in chunks, filter immediately, and collect
chunks = []
chunk_size = 500_000
rows_read = 0

reader = pd.read_csv(
    '2024_lar.txt',
    sep='|',
    dtype=str,
    chunksize=chunk_size
)

for i, chunk in enumerate(reader):
    # Filter to action_taken = 1 (originated), 2 (approved not accepted), 3 (denied)
    filtered = chunk[chunk['action_taken'].isin(['1', '2', '3'])]
    chunks.append(filtered)
    
    rows_read += len(chunk)
    print(f"  Chunk {i+1}: read {len(chunk):,} rows, kept {len(filtered):,} (running total read: {rows_read:,})")

# Combine all filtered chunks
df_raw = pd.concat(chunks, ignore_index=True)

print(f"\n{'='*50}")
print(f"Total rows in original file: ~{rows_read:,}")
print(f"Rows after filtering to action_taken 1, 2, 3: {len(df_raw):,}")
print(f"Number of columns: {df_raw.shape[1]}")
print(f"Memory usage: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Quick check on action_taken distribution
print(f"\nAction Taken distribution:")
print(df_raw['action_taken'].value_counts().sort_index())

df_raw.head(3)

  Chunk 1: read 500,000 rows, kept 380,536 (running total read: 500,000)
  Chunk 2: read 500,000 rows, kept 358,052 (running total read: 1,000,000)
  Chunk 3: read 500,000 rows, kept 361,183 (running total read: 1,500,000)
  Chunk 4: read 500,000 rows, kept 387,421 (running total read: 2,000,000)
  Chunk 5: read 500,000 rows, kept 370,531 (running total read: 2,500,000)
  Chunk 6: read 500,000 rows, kept 266,673 (running total read: 3,000,000)
  Chunk 7: read 500,000 rows, kept 394,539 (running total read: 3,500,000)
  Chunk 8: read 500,000 rows, kept 413,552 (running total read: 4,000,000)
  Chunk 9: read 500,000 rows, kept 371,482 (running total read: 4,500,000)
  Chunk 10: read 500,000 rows, kept 470,483 (running total read: 5,000,000)
  Chunk 11: read 500,000 rows, kept 346,192 (running total read: 5,500,000)
  Chunk 12: read 500,000 rows, kept 342,036 (running total read: 6,000,000)
  Chunk 13: read 500,000 rows, kept 252,943 (running total read: 6,500,000)
  Chunk 14: read 500,00

,activity_year,lei,derived_msa_md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,denial_reason_2,denial_reason_3,denial_reason_4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,2024,5493006P5IHSK77SZ855,19124,TX,48139,48139060803,C,FHA:First Lien,Single Family (1-4 Units):Site-Built,Hispanic or Latino,...,NaN,NaN,NaN,1903,51.87,110300,94.0,413,563,30
1,2024,5493006P5IHSK77SZ855,19124,TX,48139,48139060103,C,FHA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,4681,57.66,110300,93.0,1472,1628,27
2,2024,5493006P5IHSK77SZ855,19124,TX,48113,48113008703,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,3479,98.42,110300,36.0,657,1118,62


### Step 2 | Target Variable Creation and Stratified Sampling
We created a binary target variable:
- Approved, where a value of 1 indicates that the application was approved (action_taken = 1 or 2)
- Denied, where a value of 0 indicates denial (action_taken = 3)

Applications coded as "approved but not accepted" were grouped with originated loans because both reflect a positive lending decision. The distinction between the two is driven by borrower behavior after the decision, not by the lender's assessment.

With 8.66 million rows remaining, the dataset was still larger than necessary for our analysis and computationally difficult to process. We drew a stratified random sample of 500,000 rows using scikit-learn's train_test_split function, stratifying on the target variable to preserve the original approval rate of 75.72%. The full dataset was then removed from memory to ensure smooth execution of subsequent cleaning steps.

In [2]:
## Step 2: Create Target Variable & Take Stratified Sample
# Create binary target: Approved (1) vs Denied (0).
# Then take a stratified random sample of 500k rows to make the rest of the cleaning pipeline manageable in memory.

# Create binary target variable:
# Approved = action_taken 1 (originated) or 2 (approved but not accepted)
# Denied = action_taken 3
df_raw['approved'] = df_raw['action_taken'].isin(['1', '2']).astype(int)

print("Target variable distribution (full dataset):")
print(df_raw['approved'].value_counts())
print(f"\nApproval rate: {df_raw['approved'].mean():.4f} ({df_raw['approved'].mean()*100:.2f}%)")

# Take stratified random sample of 500,000 rows preserving target distribution
from sklearn.model_selection import train_test_split

df, _ = train_test_split(
    df_raw, 
    train_size=500_000, 
    random_state=42, 
    stratify=df_raw['approved']
)

df = df.reset_index(drop=True)

# Free up memory from the full dataset
del df_raw
import gc
gc.collect()

print(f"\n{'='*50}")
print(f"Sampled dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nTarget distribution in sample:")
print(df['approved'].value_counts())
print(f"\nApproval rate in sample: {df['approved'].mean():.4f} ({df['approved'].mean()*100:.2f}%)")

Target variable distribution (full dataset):
approved
1    6558326
0    2103435
Name: count, dtype: int64

Approval rate: 0.7572 (75.72%)

Sampled dataset shape: (500000, 100)
Memory usage: 2307.2 MB

Target distribution in sample:
approved
1    378579
0    121421
Name: count, dtype: int64

Approval rate in sample: 0.7572 (75.72%)


### Step 3 | Column Selection
The dataset contained 99 columns, many of which were not relevant to our research questions. We selected 32 columns organized into the following categories:

- Loan characteristics: loan type, loan purpose, preapproval status, loan amount, loan term, interest rate, lien status
- Applicant financial characteristics: income, debt-to-income ratio, combined loan-to-value ratio, and credit scoring model used
- Property characteristics: property value, occupancy type, construction method, total units
- Demographic variables: derived race, derived ethnicity, derived sex 
- Loan feature flags: balloon payment, interest-only payment, negative amortization, other non-amortizing features
- Application and underwriting details: reverse mortgage status, open-end line of credit status, business or commercial purpose, automated underwriting system used, conforming loan limit status
- Geography: state code

We excluded co-applicant fields, census tract-level demographic variables and redundant identifiers. This reduced the dataset from 99 columns to 32.

In [3]:
# Step 3: Select Relevant Columns
# We keep columns relevant to our research questions
# We drop co-applicant fields, redundant columns, and census-level variables

# Select columns to keep
cols_to_keep = [
    # Target
    'approved',
    
    # Loan characteristics
    'loan_type', 'loan_purpose', 'preapproval', 'loan_amount', 'loan_term',
    'interest_rate', 'lien_status', 'construction_method',
    
    # Applicant financial characteristics
    'income', 'debt_to_income_ratio', 'combined_loan_to_value_ratio',
    'applicant_credit_score_type',
    
    # Property characteristics
    'property_value', 'occupancy_type', 'total_units',
    
    # Applicant demographics (for fairness analysis - RQ3)
    'derived_race', 'derived_ethnicity', 'derived_sex',
    'applicant_age',
    
    # Loan features
    'balloon_payment', 'interest_only_payment', 'negative_amortization',
    'other_nonamortizing_features',
    
    # Application details
    'reverse_mortgage', 'open_end_line_of_credit',
    'business_or_commercial_purpose',
    
    # Derived fields
    'derived_loan_product_type', 'conforming_loan_limit',
    
    # Geography
    'state_code',
    
    # Underwriting
    'aus_1',
    
    # Original action_taken (for reference)
    'action_taken'
]

# Check which columns exist in the dataframe (in case of naming differences)
missing_cols = [c for c in cols_to_keep if c not in df.columns]
if missing_cols:
    print(f"\nWARNING - These columns were not found: {missing_cols}")
    print("\nSearching for similar column names...")
    for mc in missing_cols:
        similar = [c for c in df.columns if mc.split('_')[0] in c.lower()]
        print(f"  '{mc}' -> possible matches: {similar[:5]}")

# Keep only existing columns
cols_existing = [c for c in cols_to_keep if c in df.columns]
df = df[cols_existing]

print(f"\n{'='*50}")
print(f"Columns kept: {len(df.columns)}")
print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nKept columns:")
for col in df.columns:
    print(f"  - {col}")


Columns kept: 32
Dataset shape: (500000, 32)
Memory usage: 816.7 MB

Kept columns:
  - approved
  - loan_type
  - loan_purpose
  - preapproval
  - loan_amount
  - loan_term
  - interest_rate
  - lien_status
  - construction_method
  - income
  - debt_to_income_ratio
  - combined_loan_to_value_ratio
  - applicant_credit_score_type
  - property_value
  - occupancy_type
  - total_units
  - derived_race
  - derived_ethnicity
  - derived_sex
  - applicant_age
  - balloon_payment
  - interest_only_payment
  - negative_amortization
  - other_nonamortizing_features
  - reverse_mortgage
  - open_end_line_of_credit
  - business_or_commercial_purpose
  - derived_loan_product_type
  - conforming_loan_limit
  - state_code
  - aus_1
  - action_taken


## Step 4 | Data Exploration
Before cleaning, we examined each column for data types, the number of unique values and the prevalence of special codes used. This exploration revealed several important patterns:

- The code 1111 appeared across multiple columns (e.g., balloon payment, interest-only payment, reverse mortgage, automated underwriting system), representing records filed by institutions exempt from reporting these fields
- The code 8888 in the applicant age field indicated that age information was not applicable
- The string "Exempt" appeared in fields such as debt-to-income ratio and combined loan-to-value ratio
- Several columns that should be numeric (e.g., loan amount, income, property value) were stored as strings due to the presence of these special codes
- The debt-to-income ratio field contained a mix of categorical bins (e.g., "<20%", "20%-<30%") and exact numeric values between 36 and 49

In [4]:
# Step 4: Explore Data - Types, Missing Values & Special Codes
# Check unique values, missing patterns, and special codes (NA, Exempt, 1111, 8888) across all columns.

# Summary of each column
print(f"{'Column':<35} {'Dtype':<8} {'Unique':>7} {'Blanks':>7} {'NA_str':>7} {'Exempt':>7} {'1111':>7}")
print("-" * 90)

for col in df.columns:
    n_unique = df[col].nunique()
    n_blank = (df[col].isna() | (df[col] == '') | (df[col] == ' ')).sum()
    n_na_str = (df[col] == 'NA').sum()
    n_exempt = (df[col] == 'Exempt').sum()
    n_1111 = (df[col] == '1111').sum()
    print(f"{col:<35} {str(df[col].dtype):<8} {n_unique:>7,} {n_blank:>7,} {n_na_str:>7,} {n_exempt:>7,} {n_1111:>7,}")

# Show unique values for key categorical columns
print(f"\n{'='*50}")
print("Unique values for key columns:\n")

cat_cols = [
    'loan_type', 'loan_purpose', 'preapproval', 'occupancy_type',
    'construction_method', 'lien_status', 'applicant_age',
    'debt_to_income_ratio', 'total_units', 'derived_race',
    'derived_ethnicity', 'derived_sex', 'balloon_payment',
    'interest_only_payment', 'negative_amortization',
    'reverse_mortgage', 'open_end_line_of_credit',
    'business_or_commercial_purpose', 'conforming_loan_limit',
    'applicant_credit_score_type', 'aus_1'
]

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())

Column                              Dtype     Unique  Blanks  NA_str  Exempt    1111
------------------------------------------------------------------------------------------
approved                            int64          2       0       0       0       0
loan_type                           object         4       0       0       0       0
loan_purpose                        object         6       0       0       0       0
preapproval                         object         2       0       0       0       0
loan_amount                         object       892       0       0       0       0
loan_term                           object       364   9,367       0  14,656       0
interest_rate                       object     4,498 120,602       0  14,683       0
lien_status                         object         2       0       0       0       0
construction_method                 object         2       0       0       0       0
income                              object     2,823  31,59

## Step 5 | Special Code Replacement and Type Conversion
We systematically replaced special codes with NaN to standardize the representation of missing or inapplicable data. Specifically, the value 1111 was replaced with NaN in all columns where it appeared, 8888 was replaced in the applicant age field and the string Exempt was replaced in the debt-to-income ratio, combined loan-to-value ratio, interest rate, loan term and property value fields. Following these replacements, all columns were converted to their appropriate numeric data types.

In [5]:
# Step 5: Clean Special Codes & Convert Numeric Fields
# Handle 1111 (Exempt), 8888 (NA), Exempt strings, and blank values. Convert numeric columns to proper types.

print(f"Shape before cleaning: {df.shape}")

# 1. Replace '1111' with NaN across all columns where it appears
cols_with_1111 = [
    'balloon_payment', 'interest_only_payment', 'negative_amortization',
    'other_nonamortizing_features', 'reverse_mortgage', 'open_end_line_of_credit',
    'business_or_commercial_purpose', 'aus_1', 'applicant_credit_score_type'
]
for col in cols_with_1111:
    df[col] = df[col].replace('1111', np.nan)
    
print("1. Replaced '1111' (Exempt) with NaN")

# 2. Replace '8888' in applicant_age with NaN
df['applicant_age'] = df['applicant_age'].replace('8888', np.nan)
print("2. Replaced '8888' in applicant_age with NaN")

# 3. Replace 'Exempt' string with NaN
cols_with_exempt = ['debt_to_income_ratio', 'combined_loan_to_value_ratio',
                    'interest_rate', 'loan_term', 'property_value']
for col in cols_with_exempt:
    df[col] = df[col].replace('Exempt', np.nan)
    
print("3. Replaced 'Exempt' string with NaN")

# 4. Convert purely numeric columns to proper types
# These columns should be numeric (integers)
int_cols = ['loan_type', 'loan_purpose', 'preapproval', 'occupancy_type',
            'construction_method', 'lien_status', 'balloon_payment',
            'interest_only_payment', 'negative_amortization',
            'other_nonamortizing_features', 'reverse_mortgage',
            'open_end_line_of_credit', 'business_or_commercial_purpose',
            'applicant_credit_score_type', 'aus_1']

for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# These columns should be float
float_cols = ['loan_amount', 'interest_rate', 'combined_loan_to_value_ratio',
              'property_value', 'income']

for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# loan_term to numeric
df['loan_term'] = pd.to_numeric(df['loan_term'], errors='coerce')

print("4. Converted numeric columns to proper types")

# 5. Summary of missing values after cleaning
print(f"\n{'='*50}")
print("Missing values after cleaning:\n")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Pct': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Pct', ascending=False)
print(missing_df.to_string())

print(f"\nShape after cleaning: {df.shape}")

Shape before cleaning: (500000, 32)
1. Replaced '1111' (Exempt) with NaN
2. Replaced '8888' in applicant_age with NaN
3. Replaced 'Exempt' string with NaN
4. Converted numeric columns to proper types

Missing values after cleaning:

                                Missing    Pct
interest_rate                    135285  27.06
debt_to_income_ratio              54778  10.96
combined_loan_to_value_ratio      52860  10.57
property_value                    33360   6.67
income                            31596   6.32
loan_term                         24023   4.80
aus_1                             14858   2.97
balloon_payment                   14746   2.95
negative_amortization             14746   2.95
other_nonamortizing_features      14746   2.95
reverse_mortgage                  14728   2.95
interest_only_payment             14746   2.95
applicant_credit_score_type       14697   2.94
business_or_commercial_purpose    14696   2.94
open_end_line_of_credit           14659   2.93
applicant_age  

## Step 6 | Handling Missing Values
Missing values were addressed through a three-part strategy:
- Dropping exempt institution rows: Approximately 14,800 rows were missing values simultaneously across many columns due to institutional reporting exemptions. Because these rows lacked data across a wide range of features, they were dropped entirely.
- Creating missingness indicator flags: For three columns with substantial missing rates (interest rate, debt-to-income ratio, and combined loan-to-value ratio) we created binary indicator variables capturing whether the value was missing. Analysis confirmed that missingness in these fields was strongly associated with the target variable. Most notably, interest rate was missing for 100% of denied applications and only 0.31% of approved applications, since denied applicants are never assigned a rate. Debt-to-income ratio and combined loan-to-value ratio were also more frequently missing for denied applications (10.6% and 16.5%, respectively) than for approved ones (7.5% and 5.0%). These flags therefore encode meaningful predictive signal.
- Dropping rows missing key required variables: Records missing values in income, property value, loan term, applicant age, state code, or conforming loan limit were dropped, as these fields are essential for analysis and could not be reliably imputed.

This removed approximately 52,000 rows, yielding a working dataset of 433,173 records.

In [6]:
# Step 6: Handle Missing Values
# Strategy:
# 1. Drop rows from exempt institutions 
# 2. For interest_rate, DTI, CLTV — create a binary "missing" indicator flag, then fill NaN
# 3. Drop rows still missing key variables (income, property_value,loan_term, applicant_age, state_code, conforming_loan_limit)

print(f"Shape before handling missing: {df.shape}")

# Step 1: Drop exempt institution rows
# These rows are missing across many columns at once.
exempt_mask = df['balloon_payment'].isna()
print(f"\n1. Rows from exempt institutions (NaN in balloon_payment): {exempt_mask.sum():,}")
df = df[~exempt_mask].reset_index(drop=True)
print(f"   Shape after dropping exempt rows: {df.shape}")

# Step 2: Create missing indicator flags for high-missingness columns
# These columns are frequently missing for denied applications
flag_cols = ['interest_rate', 'debt_to_income_ratio', 'combined_loan_to_value_ratio']

for col in flag_cols:
    flag_name = f'{col}_missing'
    df[flag_name] = df[col].isna().astype(int)
    n_missing = df[flag_name].sum()
    print(f"\n2. Created '{flag_name}': {n_missing:,} missing ({n_missing/len(df)*100:.2f}%)")

# Check: are these more likely to be missing for denied applications?
print(f"\n   Missing rate by approval status:")
for col in flag_cols:
    flag_name = f'{col}_missing'
    print(f"\n   {col}:")
    print(f"     Approved (1): {df[df['approved']==1][flag_name].mean()*100:.2f}% missing")
    print(f"     Denied   (0): {df[df['approved']==0][flag_name].mean()*100:.2f}% missing")

# Step 3: Drop rows still missing key required variables
key_cols = ['income', 'property_value', 'loan_term', 'applicant_age',
            'state_code', 'conforming_loan_limit']

print(f"\n3. Missing in key columns before dropping:")
for col in key_cols:
    n = df[col].isna().sum()
    print(f"   {col}: {n:,} ({n/len(df)*100:.2f}%)")

df = df.dropna(subset=key_cols).reset_index(drop=True)
print(f"\n   Shape after dropping rows missing key variables: {df.shape}")

# Final missing value summary
print(f"\n{'='*50}")
print("Remaining missing values:")
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print("   None (except columns we'll handle via flags)")
else:
    for col, n in missing.items():
        print(f"   {col}: {n:,} ({n/len(df)*100:.2f}%)")

print(f"\nTarget distribution after cleaning:")
print(df['approved'].value_counts())
print(f"Approval rate: {df['approved'].mean()*100:.2f}%")

Shape before handling missing: (500000, 32)

1. Rows from exempt institutions (NaN in balloon_payment): 14,746
   Shape after dropping exempt rows: (485254, 32)

2. Created 'interest_rate_missing': 120,583 missing (24.85%)

2. Created 'debt_to_income_ratio_missing': 40,068 missing (8.26%)

2. Created 'combined_loan_to_value_ratio_missing': 38,141 missing (7.86%)

   Missing rate by approval status:

   interest_rate:
     Approved (1): 0.31% missing
     Denied   (0): 100.00% missing

   debt_to_income_ratio:
     Approved (1): 7.49% missing
     Denied   (0): 10.60% missing

   combined_loan_to_value_ratio:
     Approved (1): 5.03% missing
     Denied   (0): 16.52% missing

3. Missing in key columns before dropping:
   income: 28,973 (5.97%)
   property_value: 18,638 (3.84%)
   loan_term: 9,349 (1.93%)
   applicant_age: 10,830 (2.23%)
   state_code: 2,468 (0.51%)
   conforming_loan_limit: 1,708 (0.35%)

   Shape after dropping rows missing key variables: (433173, 35)

Remaining missin

## Step 7 | Feature Engineering and Imputation
- Interest rate removal: Because interest rate was missing for all denied applications, including it as a feature would allow a model to predict the target variable. The column was therefore dropped. The missingness indicator flag (interest_rate_missing) was retained, as it captures a legitimate data pattern without directly leaking the outcome
- Debt-to-income ratio encoding: The debt-to-income ratio field contained a mix of categorical bins and exact numeric values due to HMDA privacy modification rules: values below 36% and above 49% are reported in bins, while values between 36% and 49% are reported exactly. We unified these into a single six-level ordinal variable: 1 (<20%), 2 (20%-<30%), 3 (30%-<36%), 4 (36%-<50%), 5 (50%-60%), and 6 (>60%). Remaining missing values were imputed with the median ordinal category (4)
- Applicant age encoding: Age, reported in seven bins per HMDA privacy rules (<25, 25-34, 35-44, 45-54, 55-64, 65-74, >74), was encoded as an ordinal variable from 1 to 7
- Combined loan-to-value ratio imputation: The remaining missing values in the combined loan-to-value ratio (approximately 4.4%) were imputed with the column median of 79.00
- Minor categorical imputation: A small number of remaining missing values in applicant credit score type, reverse mortgage, business or commercial purpose and automated underwriting system (fewer than 10 rows each) were imputed with the column mode

In [7]:
# Step 7: Handle Remaining Missing, DTI Encoding & Drop Leaky Features
# 1. Drop interest_rate (leaks target — 100% missing for denied)
# 2. Encode debt_to_income_ratio as ordinal (mixed bins + exact values)
# 3. Fill remaining NaN in CLTV with median
# 4. Fill tiny remaining NaN in categorical cols with mode
# 5. Encode applicant_age as ordinal

print(f"Shape before: {df.shape}")

# 1. Drop interest_rate (target leaker)
# Keep the missing flag we already created, but drop the actual rate
df = df.drop(columns=['interest_rate'])
print("1. Dropped 'interest_rate' (100% missing for denied — would leak the target)")

# 2. Encode debt_to_income_ratio as ordinal
# The DTI field has bins (<20%, 20%-<30%, 30%-<36%, 50%-60%, >60%) and exact values between 36-49. We unify into ordinal categories.

# First, map exact values (36-49) into a single category
def categorize_dti(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if val == '<20%':
        return 1
    elif val == '20%-<30%':
        return 2
    elif val == '30%-<36%':
        return 3
    elif val in ('50%-60%',):
        return 5
    elif val == '>60%':
        return 6
    else:
        # Try to parse exact values (36-49)
        try:
            exact = float(val)
            if 36 <= exact < 50:
                return 4  # 36%-<50% category
            else:
                return np.nan
        except ValueError:
            return np.nan

df['dti_ordinal'] = df['debt_to_income_ratio'].apply(categorize_dti)

print("\n2. Encoded debt_to_income_ratio as ordinal:")
print("   1 = <20%")
print("   2 = 20%-<30%")
print("   3 = 30%-<36%")
print("   4 = 36%-<50% (exact values)")
print("   5 = 50%-60%")
print("   6 = >60%")
print(f"\n   Distribution:")
print(df['dti_ordinal'].value_counts().sort_index().to_string())
print(f"   NaN: {df['dti_ordinal'].isna().sum():,}")

# Drop original DTI column
df = df.drop(columns=['debt_to_income_ratio'])

# 3. Encode applicant_age as ordinal
age_map = {'<25': 1, '25-34': 2, '35-44': 3, '45-54': 4,
           '55-64': 5, '65-74': 6, '>74': 7}
df['age_ordinal'] = df['applicant_age'].map(age_map)
print(f"\n3. Encoded applicant_age as ordinal (1=<25 ... 7=>74)")
print(f"   Distribution:")
print(df['age_ordinal'].value_counts().sort_index().to_string())
print(f"   NaN: {df['age_ordinal'].isna().sum():,}")

# Drop original age column
df = df.drop(columns=['applicant_age'])

# 4. Fill CLTV missing with median
cltv_median = df['combined_loan_to_value_ratio'].median()
df['combined_loan_to_value_ratio'] = df['combined_loan_to_value_ratio'].fillna(cltv_median)
print(f"\n4. Filled CLTV NaN with median ({cltv_median:.2f})")

# 5. Fill DTI ordinal missing with median
dti_median = df['dti_ordinal'].median()
df['dti_ordinal'] = df['dti_ordinal'].fillna(dti_median)
print(f"5. Filled dti_ordinal NaN with median ({dti_median})")

# 6. Fill tiny remaining NaN in categorical columns with mode
tiny_nan_cols = ['applicant_credit_score_type', 'reverse_mortgage',
                 'business_or_commercial_purpose', 'aus_1']
for col in tiny_nan_cols:
    mode_val = df[col].mode()[0]
    n_fill = df[col].isna().sum()
    df[col] = df[col].fillna(mode_val)
    print(f"6. Filled {col} NaN ({n_fill} rows) with mode ({mode_val})")

# Final check
print(f"\n{'='*50}")
print(f"Shape after cleaning: {df.shape}")
print(f"\nRemaining missing values:")
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print("   NONE — dataset is fully clean!")
else:
    print(missing.to_string())

print(f"\nColumn dtypes:")
print(df.dtypes.to_string())

Shape before: (433173, 35)
1. Dropped 'interest_rate' (100% missing for denied — would leak the target)

2. Encoded debt_to_income_ratio as ordinal:
   1 = <20%
   2 = 20%-<30%
   3 = 30%-<36%
   4 = 36%-<50% (exact values)
   5 = 50%-60%
   6 = >60%

   Distribution:
dti_ordinal
1.0     24267
2.0     57500
3.0     59511
4.0    208689
5.0     43635
6.0     30475
   NaN: 9,096

3. Encoded applicant_age as ordinal (1=<25 ... 7=>74)
   Distribution:
age_ordinal
1     15856
2     87909
3    105902
4     91908
5     72269
6     42456
7     16873
   NaN: 0

4. Filled CLTV NaN with median (79.00)
5. Filled dti_ordinal NaN with median (4.0)
6. Filled applicant_credit_score_type NaN (4 rows) with mode (1.0)
6. Filled reverse_mortgage NaN (2 rows) with mode (2.0)
6. Filled business_or_commercial_purpose NaN (5 rows) with mode (2.0)
6. Filled aus_1 NaN (6 rows) with mode (6.0)

Shape after cleaning: (433173, 34)

Remaining missing values:
   NONE — dataset is fully clean!

Column dtypes:
approved

## Step 8 | Final Categorical Encoding and Cleanup
- The total units field was encoded as an ordinal variable from 1 to 9, preserving the natural ordering from single-unit to 149+ unit properties
- Conforming loan limit was encoded as a three-level variable (1 = Conforming, 2 = Non-conforming, 3 = Undetermined)
- The derived_loan_product_type column was dropped because it was a composite of loan type and lien status, both of which were already included as separate features
- The original action_taken column was also dropped as it was fully captured by the binary approved target variable
- All float columns that represented integer categories were converted to integer type

In [8]:
# Step 8: Encode Remaining Categoricals & Final Cleanup
# Remaining object columns to handle:
# - total_units: ordinal (1,2,3,4 exact + bins above 4)
# - derived_race, derived_ethnicity, derived_sex: keep as-is for fairness analysis (RQ3)
# - derived_loan_product_type: high cardinality — review
# - conforming_loan_limit: C, NC, U — simple categorical
# - state_code: keep as-is for geographic grouping
# - action_taken: reference only — we already have 'approved'

print(f"Shape: {df.shape}")
print(f"\nObject columns remaining:")
obj_cols = df.select_dtypes(include='object').columns.tolist()
for col in obj_cols:
    print(f"  {col}: {df[col].nunique()} unique values")

# 1. Encode total_units as ordinal
units_map = {'1': 1, '2': 2, '3': 3, '4': 4,
             '5-24': 5, '25-49': 6, '50-99': 7, '100-149': 8, '>149': 9}
df['total_units'] = df['total_units'].map(units_map)
print(f"\n1. Encoded total_units as ordinal (1-9)")

# 2. Encode conforming_loan_limit
# C = Conforming, NC = Non-conforming, U = Undetermined
climit_map = {'C': 1, 'NC': 2, 'U': 3}
df['conforming_loan_limit'] = df['conforming_loan_limit'].map(climit_map)
print("2. Encoded conforming_loan_limit (1=C, 2=NC, 3=U)")

# 3. Review derived_loan_product_type
print(f"\n3. derived_loan_product_type distribution:")
print(df['derived_loan_product_type'].value_counts().to_string())

# 4. Review demographic columns
print(f"\n4. Demographic columns (for RQ3 fairness analysis):")
for col in ['derived_race', 'derived_ethnicity', 'derived_sex']:
    print(f"\n   --- {col} ---")
    vc = df[col].value_counts()
    for val, count in vc.items():
        print(f"   {val}: {count:,} ({count/len(df)*100:.1f}%)")

# 5. Drop action_taken (we have 'approved' as our target)
df = df.drop(columns=['action_taken'])
print(f"\n5. Dropped 'action_taken' (redundant — we have 'approved')")

# 6. Convert float columns that should be int
float_to_int = ['loan_term', 'balloon_payment', 'interest_only_payment',
                'negative_amortization', 'other_nonamortizing_features',
                'reverse_mortgage', 'open_end_line_of_credit',
                'business_or_commercial_purpose', 'applicant_credit_score_type',
                'aus_1', 'dti_ordinal']
for col in float_to_int:
    df[col] = df[col].astype(int)
print("6. Converted float columns to int where appropriate")

# Final Summary
print(f"\n{'='*50}")
print(f"Final cleaned dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nColumn types summary:")
print(df.dtypes.value_counts())
print(f"\nAll columns:")
for col in df.columns:
    print(f"  {col}: {df[col].dtype}")

Shape: (433173, 34)

Object columns remaining:
  total_units: 4 unique values
  derived_race: 9 unique values
  derived_ethnicity: 5 unique values
  derived_sex: 4 unique values
  derived_loan_product_type: 8 unique values
  conforming_loan_limit: 3 unique values
  state_code: 54 unique values
  action_taken: 3 unique values

1. Encoded total_units as ordinal (1-9)
2. Encoded conforming_loan_limit (1=C, 2=NC, 3=U)

3. derived_loan_product_type distribution:
derived_loan_product_type
Conventional:First Lien          223479
Conventional:Subordinate Lien    122397
FHA:First Lien                    57927
VA:First Lien                     26952
FSA/RHS:First Lien                 2163
FSA/RHS:Subordinate Lien            243
FHA:Subordinate Lien                 10
VA:Subordinate Lien                   2

4. Demographic columns (for RQ3 fairness analysis):

   --- derived_race ---
   White: 285,557 (65.9%)
   Race Not Available: 66,364 (15.3%)
   Black or African American: 38,042 (8.8%)
   Asi

## Step 9 | Export
The final cleaned dataset contains 433,173 records and 32 columns with zero missing values. The target variable maintains an approval rate of 77.36%, consistent with the original full dataset rate of 75.72%. The dataset was exported as hmda_2024_cleaned.csv for use in subsequent modeling and analysis.

The 32 columns in the final dataset are organized as follows:
- 1 target variable
- 7 numeric features
- 17 categorical features encoded as integers
- 3 missingness indicator flags
- 3 demographic columns for fairness analysis
- 1 geographic identifier

In [9]:
# Step 9: Final Checks & Export Cleaned Dataset

# 1. Drop derived_loan_product_type (redundant with loan_type + lien_status)
df = df.drop(columns=['derived_loan_product_type'])
print("1. Dropped 'derived_loan_product_type' (redundant with loan_type + lien_status)")

# 2. Sanity checks
print(f"\n{'='*50}")
print("FINAL DATASET SUMMARY")
print(f"{'='*50}")
print(f"\nShape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\nTarget variable:")
print(f"  Approved (1): {(df['approved']==1).sum():,} ({df['approved'].mean()*100:.2f}%)")
print(f"  Denied   (0): {(df['approved']==0).sum():,} ({(1-df['approved'].mean())*100:.2f}%)")

print(f"\nNumeric columns - descriptive stats:")
num_cols = ['loan_amount', 'loan_term', 'income', 'property_value',
            'combined_loan_to_value_ratio', 'dti_ordinal', 'age_ordinal']
print(df[num_cols].describe().round(2).to_string())

print(f"\nCategorical columns (object type):")
for col in df.select_dtypes(include='object').columns:
    print(f"\n  {col} ({df[col].nunique()} categories):")
    print(f"  {df[col].value_counts().head(5).to_string()}")

# 3. List all columns by role
print(f"\n{'='*50}")
print("COLUMNS BY ROLE:")
print(f"\nTarget (1):")
print(f"  approved")

print(f"\nNumeric features ({len(num_cols)}):")
for c in num_cols:
    print(f"  {c}")

cat_int_cols = ['loan_type', 'loan_purpose', 'preapproval', 'lien_status',
                'construction_method', 'occupancy_type', 'total_units',
                'conforming_loan_limit', 'applicant_credit_score_type',
                'balloon_payment', 'interest_only_payment', 'negative_amortization',
                'other_nonamortizing_features', 'reverse_mortgage',
                'open_end_line_of_credit', 'business_or_commercial_purpose', 'aus_1']
print(f"\nCategorical features encoded as int ({len(cat_int_cols)}):")
for c in cat_int_cols:
    print(f"  {c}")

flag_cols = ['interest_rate_missing', 'debt_to_income_ratio_missing',
             'combined_loan_to_value_ratio_missing']
print(f"\nMissingness indicator flags ({len(flag_cols)}):")
for c in flag_cols:
    print(f"  {c}")

demo_cols = ['derived_race', 'derived_ethnicity', 'derived_sex']
print(f"\nDemographic columns for fairness analysis ({len(demo_cols)}):")
for c in demo_cols:
    print(f"  {c}")

print(f"\nGeographic:")
print(f"  state_code")

# 4. Export
df.to_csv('hmda_2024_cleaned.csv', index=False)
print(f"\n{'='*50}")
print("Exported cleaned dataset to 'hmda_2024_cleaned.csv'")
print(f"File ready for analysis!")

1. Dropped 'derived_loan_product_type' (redundant with loan_type + lien_status)

FINAL DATASET SUMMARY

Shape: (433173, 32)
Missing values: 0
Memory: 198.4 MB

Target variable:
  Approved (1): 335,087 (77.36%)
  Denied   (0): 98,086 (22.64%)

Numeric columns - descriptive stats:
       loan_amount  loan_term     income  property_value  combined_loan_to_value_ratio  dti_ordinal  age_ordinal
count    433173.00  433173.00  433173.00    4.331730e+05                     433173.00    433173.00    433173.00
mean     271710.30     321.80     152.02    5.189331e+05                         92.62         3.67         3.72
std      297327.99      78.48     598.21    4.784919e+06                       7641.62         1.20         1.49
min        5000.00       1.00  -42909.00    5.000000e+03                          0.01         1.00         1.00
25%       95000.00     300.00      69.00    2.550000e+05                         61.08         3.00         3.00
50%      205000.00     360.00     105.00  